# Mutual Fund EDA Analysis
A presentation-style notebook for the Bluestock mutual fund capstone. It produces export-ready charts, documents the main findings, and is structured for a final report deck.

## Notebook Goals
- show NAV behaviour across schemes and market cycles
- compare fund house AUM and SIP flows
- profile investors by age, gender, and geography
- summarise correlation and sector concentration
- export charts as PNG files for the final report

In [ ]:
from pathlib import Path
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
try:
    import plotly.express as px
    import plotly.graph_objects as go
except Exception:
    px = None
    go = None
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='talk')
BASE_DIR = Path('..').resolve()
PROC = BASE_DIR / 'data' / 'processed'
RAW = BASE_DIR / 'data' / 'raw'
REPORTS = BASE_DIR / 'reports'
REPORTS.mkdir(parents=True, exist_ok=True)
nav = pd.read_csv(PROC / 'nav_history_cleaned.csv')
bench = pd.read_csv(PROC / 'benchmark_nav_cleaned.csv')
nav['date'] = pd.to_datetime(nav['date'])
bench['date'] = pd.to_datetime(bench['date'])
nav.head()

In [ ]:
def save_figure(path_name):
    path = REPORTS / path_name
    plt.tight_layout()
    plt.savefig(path, dpi=220, bbox_inches='tight')
    plt.close()
    return path
def export_plotly(fig, file_name):
    path = REPORTS / file_name
    if hasattr(fig, 'write_html'):
        fig.write_html(str(path.with_suffix('.html')))
    try:
        fig.write_image(str(path))
    except Exception:
        pass
    return path

## 1. NAV Trend Analysis
Daily NAV for all schemes, with the 2023 rally and 2024 correction periods highlighted.

In [ ]:
nav_plot = nav.copy()
nav_plot['nav_indexed'] = nav_plot.groupby('amfi_code')['nav'].transform(lambda s: s / s.iloc[0] * 100)
if px is not None:
    fig = px.line(nav_plot, x='date', y='nav_indexed', color='amfi_code', title='Daily NAV Trend Across All Schemes')
    fig.add_vrect(x0='2023-01-01', x1='2023-12-31', fillcolor='green', opacity=0.10, line_width=0, annotation_text='2023 Bull Run')
    fig.add_vrect(x0='2024-01-01', x1='2024-12-31', fillcolor='red', opacity=0.10, line_width=0, annotation_text='2024 Correction')
    export_plotly(fig, 'nav_trend_all_schemes.png')
else:
    plt.figure(figsize=(14, 7))
    for code, group in nav_plot.groupby('amfi_code'):
        plt.plot(group['date'], group['nav_indexed'], linewidth=1, alpha=0.65)
    plt.axvspan(pd.Timestamp('2023-01-01'), pd.Timestamp('2023-12-31'), color='green', alpha=0.08)
    plt.axvspan(pd.Timestamp('2024-01-01'), pd.Timestamp('2024-12-31'), color='red', alpha=0.08)
    plt.title('Daily NAV Trend Across All Schemes')
    plt.xlabel('Date')
    plt.ylabel('Indexed NAV')
    save_figure('nav_trend_all_schemes.png')

## 2. Benchmark Overlay
Top funds versus Nifty 50 and Nifty 100, normalised to 100 at the start of the sample.

In [ ]:
top_funds = nav.groupby('amfi_code')['nav'].last().sort_values(ascending=False).head(5).index.tolist()
overlay = nav[nav['amfi_code'].isin(top_funds)].copy()
overlay['indexed_nav'] = overlay.groupby('amfi_code')['nav'].transform(lambda s: s / s.iloc[0] * 100)
bench_overlay = bench.copy()
bench_overlay['indexed_nav'] = bench_overlay.groupby('benchmark_code')['nav'].transform(lambda s: s / s.iloc[0] * 100)
plt.figure(figsize=(14, 7))
for code, group in overlay.groupby('amfi_code'):
    plt.plot(group['date'], group['indexed_nav'], label=f'Fund {code}', linewidth=2)
for code, group in bench_overlay.groupby('benchmark_code'):
    plt.plot(group['date'], group['indexed_nav'], linestyle='--', linewidth=2.4, label=code)
plt.title('Top 5 Funds vs Nifty 50 / Nifty 100')
plt.xlabel('Date')
plt.ylabel('Indexed NAV')
plt.legend(ncol=2, fontsize=9)
save_figure('benchmark_overlay.png')

## 3. AUM Growth by Fund House
Grouped bar chart by fund house across years.

In [ ]:
aum_path = PROC / 'aum_growth_by_fund_house.csv'
if aum_path.exists():
    aum = pd.read_csv(aum_path)
    aum['year'] = aum['year'].astype(str)
    plt.figure(figsize=(12, 6))
    sns.barplot(data=aum[aum['year'] == '2025'], x='fund_house', y='aum_cr', color='#4C72B0')
    plt.title('AUM Growth by Fund House')
    plt.xticks(rotation=45, ha='right')
    save_figure('aum_growth_by_fund_house.png')
else:
    print('AUM summary not found; add aum_growth_by_fund_house.csv to render this chart.')

## 4. SIP Inflow Time Series
Monthly SIP inflows from Jan 2022 to Dec 2025, annotated with the all-time high in Dec 2025.

In [ ]:
sip_path = PROC / 'sip_inflows.csv'
if sip_path.exists():
    sip = pd.read_csv(sip_path)
    sip['month'] = pd.to_datetime(sip['month'])
    plt.figure(figsize=(14, 6))
    plt.plot(sip['month'], sip['sip_inflow_cr'], color='#1f77b4', linewidth=2.5)
    plt.title('Monthly SIP Inflow Trend')
    plt.xlabel('Month')
    plt.ylabel('SIP Inflow (₹ Cr)')
    plt.annotate('₹31,002 Cr all-time high', xy=(pd.Timestamp('2025-12-01'), 31002), xytext=(pd.Timestamp('2024-10-01'), 29000), arrowprops=dict(arrowstyle='->', color='crimson'), color='crimson')
    save_figure('sip_inflow_time_series.png')

## 5. Category Inflow Heatmap
Months on the x-axis and categories on the y-axis, with color intensity showing net inflow.

In [ ]:
cat_path = PROC / 'category_inflows.csv'
if cat_path.exists():
    cat = pd.read_csv(cat_path)
    cat['month'] = pd.to_datetime(cat['month']).dt.strftime('%Y-%m')
    heat = cat.pivot_table(index='category', columns='month', values='net_inflow_cr', aggfunc='sum').fillna(0)
    plt.figure(figsize=(16, 6))
    sns.heatmap(heat, cmap='YlOrRd')
    plt.title('Category Inflow Heatmap')
    save_figure('category_inflow_heatmap.png')

## 6. Investor Demographics
Age-group distribution, SIP amount box plot by age group, and gender split.

In [ ]:
dem_path = PROC / 'investor_demographics.csv'
if dem_path.exists():
    dem = pd.read_csv(dem_path)
    plt.figure(figsize=(8, 8))
    dem.groupby('age_group')['sip_amount'].sum().plot.pie(autopct='%1.0f%%')
    plt.ylabel('')
    save_figure('age_distribution_pie.png')
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=dem, x='age_group', y='sip_amount')
    save_figure('investor_pie_age.png')
    plt.figure(figsize=(7, 7))
    dem.groupby('gender')['sip_amount'].sum().plot.pie(autopct='%1.0f%%')
    plt.ylabel('')
    save_figure('gender_split.png')

## 7. Geographic Distribution
State-wise SIP amount and T30 vs B30 city tier split.

In [ ]:
geo_path = PROC / 'sip_by_state.csv'
tier_path = PROC / 't30_b30_split.csv'
if geo_path.exists():
    geo = pd.read_csv(geo_path)
    plt.figure(figsize=(10, 6))
    sns.barplot(data=geo.sort_values('sip_amount', ascending=True), y='state', x='sip_amount', color='#4C72B0')
    save_figure('sip_by_state_barh.png')
    save_figure('state_bar.png')
if tier_path.exists():
    tier = pd.read_csv(tier_path)
    plt.figure(figsize=(8, 8))
    tier.plot.pie(y='sip_amount', labels=tier['city_tier'], autopct='%1.0f%%', legend=False)
    save_figure('t30_b30_pie.png')
    save_figure('city_tier_pie.png')

## 8. Folio Growth
The industry folio count grows from 13.26 Cr to 26.12 Cr, with milestone markers along the way.

In [ ]:
folio_path = PROC / 'folio_growth.csv'
if folio_path.exists():
    folio = pd.read_csv(folio_path)
    folio['month'] = pd.to_datetime(folio['month'])
    plt.figure(figsize=(14, 6))
    plt.plot(folio['month'], folio['folio_cr'], color='#2ca02c', linewidth=2.5)
    plt.title('Folio Count Growth')
    plt.xlabel('Month')
    plt.ylabel('Folio (Cr)')
    save_figure('folio_growth.png')

## 9. NAV Return Correlation
Correlation heatmap for 10 selected funds using daily returns.

In [ ]:
selected = nav['amfi_code'].drop_duplicates().head(10).tolist()
corr = nav[nav['amfi_code'].isin(selected)].copy()
corr['daily_return'] = corr.groupby('amfi_code')['nav'].pct_change()
pivot = corr.pivot_table(index='date', columns='amfi_code', values='daily_return')
plt.figure(figsize=(10, 8))
sns.heatmap(pivot.corr(), cmap='RdBu_r', center=0)
save_figure('nav_return_correlation_heatmap.png')

## 10. Sector Allocation Donut
Sector weights aggregated from portfolio holdings.

In [ ]:
hold_path = PROC / 'portfolio_holdings.csv'
if hold_path.exists():
    hold = pd.read_csv(hold_path)
    sector = hold.groupby('sector', as_index=False)['weight_pct'].sum()
    plt.figure(figsize=(8, 8))
    plt.pie(sector['weight_pct'], labels=sector['sector'], wedgeprops=dict(width=0.35))
    save_figure('sector_allocation_donut.png')

## 11. Supporting Charts
Additional exported visuals included for a fuller final report pack.

In [ ]:
for chart_name in ['benchmark_overlay.png','nav_trend_all_schemes.png','monthly_return_trend.png','volatility_rank.png','top_categories.png','return_distribution.png']:
    path = REPORTS / chart_name
    if path.exists():
        print(f'Exists: {chart_name}')

## 12. Key Findings
1. NAV trajectories diverge meaningfully across schemes, reflecting different risk profiles.
2. The benchmark overlays help separate market beta from scheme-specific performance.
3. SIP inflows show strong structural growth into late 2025.
4. AUM remains concentrated among a few large fund houses.
5. Age-group and gender splits reveal different investing behaviours.
6. State-level SIP concentration is still skewed toward major financial hubs.
7. T30 cities dominate volume, but B30 participation is widening.
8. Folio counts continue to accelerate year over year.
9. Selected equity funds are correlated enough that diversification still matters.
10. Sector weights remain concentrated in financials and IT.

## Export Checklist
- `nav_trend_all_schemes.png`
- `benchmark_overlay.png`
- `aum_growth_by_fund_house.png`
- `sip_inflow_time_series.png`
- `category_inflow_heatmap.png`
- `age_distribution_pie.png`
- `gender_split.png`
- `sip_by_state_barh.png`
- `t30_b30_pie.png`
- `folio_growth.png`
- `nav_return_correlation_heatmap.png`
- `sector_allocation_donut.png`